In [73]:
import pandas as pd
import numpy as np
import re
from playwright.async_api import async_playwright
import asyncio

In [102]:
USER_DATA_DIR = "/Users/irfan.hilman/Downloads/user_data_test_local"
p = await async_playwright().start()
context = await p.chromium.launch_persistent_context(
    USER_DATA_DIR,
    headless=False,
    args=["--disable-blink-features=AutomationControlled",
        "--start-minimized",
        "--disable-background-timer-throttling",
        "--disable-renderer-backgrounding",
        "--disable-backgrounding-occluded-windows",
        "--disable-features=IsolateOrigins,site-per-process",
        "--disable-gpu",
        "--disable-extensions",
        "--disable-sync",
        "--disable-default-apps"],
    viewport=None)
if context.pages:
    page = context.pages[0]
    await page.close()

In [ ]:
page_1 = await context.new_page()
await page_1.goto("https://stockbit.com/stream")

page_2 = await context.new_page()
await page_2.goto("https://stockbit.com/stream")


<Response url='https://stockbit.com/securities/portfolio' request=<Request url='https://stockbit.com/securities/portfolio' method='GET'>>

BUY OR SELL ORDER

In [100]:
stock = 'PPGL'
action = 'Buy' # Buy or Sell
order_type = 'Limit' # Market or Limit
num_lots = "20"
price = "220"

In [ ]:
# Select stock
search_stock = page_1.locator('[data-cy="top-navbar-search-input-desktop"]')
await search_stock.click()
await search_stock.fill(stock)
await search_stock.press("Enter")

# Hover to container buy/sell
container = page_1.locator('[data-overlayscrollbars-viewport]')
await container.hover()              # give focus to the scroll area
await container.evaluate("""(el) => {el.scrollTop = el.scrollHeight}""")

if action == 'Buy':
    buy_tab = page_1.get_by_role("button", name=re.compile(r"^Buy$")).nth(1)
    await buy_tab.click()
elif action == 'Sell':
    sell_tab = page_1.get_by_role("button", name="Sell").first
    await sell_tab.click()

if order_type == 'Market':
    mo_tab = page_1.get_by_role("button", name="Market").first
    await mo_tab.click()
    lot_input = page_1.locator('[data-cy="input-buy-lot"]')
    await lot_input.click()
    await lot_input.fill(num_lots)
    buy_button = page_1.get_by_role("button", name=re.compile(r"^Buy$")).nth(2)
    await asyncio.sleep(0.1)
    is_disabled = await buy_button.is_disabled()
    if not is_disabled:
        await buy_button.click()
        confirm = page_1.get_by_role("button", name="Confirm")
        await confirm.click()
        done_button = page_1.get_by_role("button", name="Return to Orderbook")
        await done_button.click()
elif order_type == 'Limit':
    lo_tab = page_1.get_by_role("button", name="Limit").first
    await lo_tab.click()
    price_input = page_1.locator('[data-cy="input-buy-price"]')
    await price_input.click()
    await price_input.fill(price)
    lot_input = page_1.locator('[data-cy="input-lot"]')
    await lot_input.click()
    await lot_input.fill(num_lots)
    buy_button = page_1.get_by_role("button", name=re.compile(r"^Buy$")).nth(2)
    await asyncio.sleep(0.1)
    if not is_disabled:
        await buy_button.click()
        confirm = page_1.get_by_role("button", name="Confirm")
        await confirm.click()
        done_button = page_1.get_by_role("button", name="Return to Orderbook")
        await done_button.click()


PORTFOLIO ANALYSIS

In [108]:
PIN = ['1','3','3','6','6','5']

In [118]:
await page_2.locator('[data-cy="navbar-portfolio"] a').first.click()

for p in PIN:
    await page_2.keyboard.type(p)
    await asyncio.sleep(0.1)

await page_2.get_by_role("button", name="Submit").click()
await asyncio.sleep(0.5)
if page_2.url == "https://stockbit.com/securities/portfolio":
    print("Already on portfolio page")

await page_2.locator('label.ant-radio-button-wrapper').filter(has_text="Order").click()

Already on portfolio page


In [ ]:
rows = await page_2.locator('[data-cy="securities-order-table"] tbody tr').all()
table_data = []
for row in rows:
    cells = await row.locator('td').all_inner_texts()
    table_data.append(cells)

df = pd.DataFrame(table_data,columns=["checkbox","Ordered Items","Status","Price","Lot Ordered","Lot Done","Amount Ordered","Amount Done","Expiry","Action"])


In [134]:
next_page = page_2.get_by_role("button", name=re.compile(r"^Next$"))
next_is_disabled = await next_page.is_disabled()
if next_is_disabled == False:
    await next_page.click()

In [135]:
prev_page = page_2.get_by_role("button", name=re.compile(r"^Prev$"))
prev_is_disabled = await prev_page.is_disabled()
if prev_is_disabled == False:
    await prev_page.click()